# 🐉 Pipeline Dragão — Imagem → 3D → Rig → Ossos nomeados
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sr-PiCINiNi/Bot-Whatsapp/blob/main/Uniring.ipynb)

**Runtime → Change runtime type → T4 GPU.**

Fluxo: **(1) imagem → malha (TripoSR)** · **(2) rig (UniRig)** · **(3) renomear ossos** · **(4) baixar `.glb`** pronto pro Roblox.

⚠️ A célula do **condacolab (2a)** REINICIA o kernel. Rode a Etapa 1, deixe reiniciar na 2a, depois siga 2b→4. Não use "Run all" de ponta a ponta.

In [ ]:
# Checagem de GPU/versoes
!nvidia-smi
import sys, torch
print("Python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU:", torch.cuda.is_available())

## Etapa 1 — Imagem → 3D (TripoSR)
Rode **1a** (instala) e **1b** (sobe a imagem). Pra usar malha do **Meshy**, pule pra **1c**.

In [ ]:
# 1a) instalar TripoSR (Python padrao do Colab, ANTES do condacolab)
%cd /content
import os
if not os.path.isdir("/content/TripoSR"):
    !git clone https://github.com/VAST-AI-Research/TripoSR
%cd /content/TripoSR
!pip install -q -r requirements.txt
!pip install -q onnxruntime
print("TripoSR instalado")

In [ ]:
# 1b) subir imagem -> gera /content/input_mesh.obj  (contorna o rembg; usa --no-remove-bg)
%cd /content/TripoSR
import pathlib
_rp = pathlib.Path("run.py")
_rp.write_text(_rp.read_text().replace("import rembg", "try:
    import rembg
except Exception:
    rembg = None", 1))
from google.colab import files
up = files.upload()
img = list(up.keys())[0]
!python run.py "{img}" --output-dir /content/tsr_out --model-save-format obj --no-remove-bg
import glob, shutil
hits = sorted(glob.glob("/content/tsr_out/**/*.obj", recursive=True))
assert hits, "TripoSR nao gerou .obj -- veja o erro acima"
shutil.copy(hits[0], "/content/input_mesh.obj")
print("malha em /content/input_mesh.obj")

In [ ]:
# 1c) (ALTERNATIVA) usar malha sua (Meshy etc.) -> /content/input_mesh.*
from google.colab import files
import shutil, os
up = files.upload()
f = list(up.keys())[0]
dst = "/content/input_mesh" + os.path.splitext(f)[1]
shutil.copy(f, dst)
print("usando", dst)

## Etapa 2 — Rig (UniRig)
**2a reinicia o kernel.** Espere reiniciar e siga 2b→2e. As correcoes de T4/flash-attn ja estao embutidas.

In [ ]:
# 2a) condacolab (Python 3.11 p/ UniRig). >>> REINICIA O KERNEL SOZINHO <<<
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# 2b) clonar UniRig + ambiente Python 3.11 + deps (5-8 min)
import os, pathlib
if not os.path.isdir("/content/UniRig"):
    !git clone https://github.com/VAST-AI-Research/UniRig
%cd /content/UniRig
!conda create -n unirig python=3.11 -y
R = "conda run -n unirig"
!{R} pip install -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
req = [l for l in pathlib.Path("requirements.txt").read_text().splitlines() if l.strip() and "flash" not in l.lower()]
pathlib.Path("requirements_nofa.txt").write_text(chr(10).join(req))
!{R} pip install -q -r requirements_nofa.txt
!{R} pip install -q numpy==1.26.4 spconv-cu120
!{R} pip install -q torch_scatter torch_cluster -f https://data.pyg.org/whl/torch-2.4.0+cu121.html --no-cache-dir
print("OK ambiente 'unirig' pronto")

In [ ]:
# 2c) flash-attn (wheel pronto p/ torch2.4 / py3.11)
!conda run -n unirig pip install -q https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3.post1/flash_attn-2.8.3.post1+cu12torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl
print("flash_attn instalado")

In [ ]:
# 2d) PATCHES p/ T4 (Turing, sem flash-attn): sdpa + MHA no-flash + PTv3 sem flash
%cd /content/UniRig
import pathlib, re, urllib.request
for f in pathlib.Path(".").rglob("*"):
    if f.is_file() and f.suffix in (".py",".yaml",".yml",".json"):
        t=f.read_text(errors="ignore")
        if "flash_attention_2" in t: f.write_text(t.replace("flash_attention_2","sdpa"))
for f in pathlib.Path("src").rglob("*.py"):
    t=f.read_text(errors="ignore")
    if re.search(r"\bMHA\(", t):
        t2=re.sub(r"\bMHA\(([^()]*)\)", lambda m: m.group(0) if "use_flash_attn" in m.group(1) else "MHA("+m.group(1)+", use_flash_attn=False)", t)
        if t2!=t: f.write_text(t2)
p=pathlib.Path("src/model/pointcept/models/PTv3Object.py")
s=urllib.request.urlopen("https://raw.githubusercontent.com/VAST-AI-Research/UniRig/main/src/model/pointcept/models/PTv3Object.py").read().decode()
s=s.replace("self.enable_flash = enable_flash","self.enable_flash = False").replace("self.patch_size_max","self.patch_size").replace("self.attn_drop(attn)","attn")
p.write_text(s)
print(">>> PATCHES APLICADOS <<<")

In [ ]:
# 2e) RODAR O RIG -> results/rigged.glb
%cd /content/UniRig
import glob, os
INPUT = sorted(glob.glob("/content/input_mesh.*"))[0]
print("entrada:", INPUT)
!rm -rf results tmp
R = "conda run -n unirig"
!{R} bash launch/inference/generate_skeleton.sh --input "{INPUT}" --output results/skeleton.fbx
!{R} bash launch/inference/generate_skin.sh     --input results/skeleton.fbx --output results/skin.fbx
!{R} bash launch/inference/merge.sh             --source results/skin.fbx --target "{INPUT}" --output results/rigged.glb
print("skeleton:", os.path.exists("results/skeleton.fbx"), "| skin:", os.path.exists("results/skin.fbx"), "| RIGGED:", os.path.exists("results/rigged.glb"))

## Etapa 3 — Renomear ossos automaticamente
Descobre cabeca/pernas/cauda/corpo pela posicao e renomeia (osso + vertex group). Imprime o mapeamento pra conferir.

In [ ]:
# 3) auto-rename por posicao (Blender do ambiente unirig)
%cd /content/UniRig
script = '\nimport bpy, sys\ninp, outp = sys.argv[-2], sys.argv[-1]\nbpy.ops.wm.read_factory_settings(use_empty=True)\nbpy.ops.import_scene.gltf(filepath=inp)\narm = next(o for o in bpy.data.objects if o.type=="ARMATURE")\nmeshes = [o for o in bpy.data.objects if o.type=="MESH"]\nbones = list(arm.data.bones)\nM = arm.matrix_world\ndef hp(b): return M @ b.head_local\ndef tp(b): return M @ b.tail_local\nH = {b.name: hp(b) for b in bones}\nxs=[p.x for p in H.values()]; ys=[p.y for p in H.values()]\nLA = "x" if (max(xs)-min(xs)) >= (max(ys)-min(ys)) else "y"\nWA = "y" if LA=="x" else "x"\ndef Lc(p): return getattr(p, LA)\ndef Wc(p): return getattr(p, WA)\nLmid = (max(Lc(p) for p in H.values()) + min(Lc(p) for p in H.values()))/2\ntop = max(bones, key=lambda b: hp(b).z)\nhead_sign = 1 if Lc(hp(top)) > Lmid else -1\nzt = [tp(b).z for b in bones]; zthr = min(zt) + (max(zt)-min(zt))*0.45\nfeet = [b for b in bones if tp(b).z < zthr]\nfeetset = set(b.name for b in feet)\ndef hip_of(b):\n    cur=b\n    while cur.parent and cur.parent.name in feetset: cur=cur.parent\n    return cur\nhips = {hip_of(b).name: hip_of(b) for b in feet}\nrenames = {}\nquadname = {(True,True):"frontleg_r",(True,False):"frontleg_l",(False,True):"backleg_r",(False,False):"backleg_l"}\nquads = {}\nfor b in hips.values():\n    front = (Lc(hp(b))-Lmid)*head_sign > 0\n    right = Wc(hp(b)) > 0\n    quads.setdefault((front,right), []).append(b)\nfor q,bs in quads.items():\n    b = max(bs, key=lambda x: hp(x).z)\n    renames[b.name] = quadname[q]\nhc = [b for b in bones if (Lc(hp(b))-Lmid)*head_sign>0 and b.name not in renames]\nhc.sort(key=lambda b: hp(b).z, reverse=True)\nfor b in hc:\n    if len(b.children)>=1: renames[b.name]="head"; break\nroot = next((b for b in bones if b.parent is None), bones[0])\nrenames.setdefault(root.name, "body")\ntc = [b for b in bones if (Lc(hp(b))-Lmid)*head_sign<0 and b.name not in renames and b not in feet]\ntc.sort(key=lambda b: abs(Lc(hp(b))-Lmid))\nfor i,b in enumerate(tc[:6],1): renames[b.name]=f"tail{i}"\nprint("=== MAPEAMENTO ===")\nfor old,new in renames.items():\n    for m in meshes:\n        vg = m.vertex_groups.get(old)\n        if vg: vg.name = new\n    arm.data.bones[old].name = new\n    print("  ", old, "->", new)\nbpy.ops.export_scene.gltf(filepath=outp, export_format="GLB")\nprint("EXPORTADO:", outp)\n'
open('rename_bones.py','w').write(script)
!conda run -n unirig python rename_bones.py results/rigged.glb /content/dragao_final.glb
import os; print('FINAL existe:', os.path.exists('/content/dragao_final.glb'))

In [ ]:
# 4) baixar o dragao final (ossos nomeados) -> Roblox: importar skinned em Assets.Dragons.<elemento>
from google.colab import files
files.download("/content/dragao_final.glb")